# Detección de retinopatía diabética — entrenamiento reproducible

**Proyecto final · Introducción a Machine Learning (UPCH)** — Edithson Ayabar, Magno Luque y Gian Carlos Quezada.

Esta es la versión reproducible del notebook original ([`01_original_upch.ipynb`](01_original_upch.ipynb)).
Conserva su enfoque —**ResNet50 preentrenada + cabeza propia, entrenada en dos etapas**— y cambia lo necesario
para que cualquiera pueda ejecutarlo y para que el modelo se pueda desplegar:

| | Original (2024) | Esta versión |
|---|---|---|
| Datos | Competencia privada `upch-intro-ml` | **APTOS 2019** (público), binarizado igual: 0 = sano, 1 = retinopatía |
| Color | Entrena en RGB, predice en BGR (`cv2`) | Un único preprocesado RGB para entrenar y predecir |
| Normalización | `rescale=1/255` | La de ResNet50, **dentro del modelo** |
| Validación | Con aumentación, solo *accuracy* | Sin aumentación; AUC, sensibilidad, especificidad, matriz de confusión |
| Salida | `modelo.h5` (se perdió con la sesión) | `modelo.onnx` con **Grad-CAM incluido** + `metrics.json` |

## Cómo ejecutarlo en Kaggle (~45 min, gratis)
1. Crea un notebook en Kaggle e importa este archivo (*File → Import notebook*).
2. *Add input* → busca la competencia **APTOS 2019 Blindness Detection** y acéptale las reglas.
3. *Settings → Accelerator → GPU T4*. Deja *Internet* activado (descarga los pesos de ImageNet).
4. *Run all*. Al terminar, descarga **`artefactos.zip`** del panel *Output* y descomprímelo en la carpeta `models/` del repositorio.

In [ ]:
# ---------- Configuración ----------
import os

SEMILLA = 42
TAMANO = 512                  # igual que en el proyecto original
LOTE = 8                      # igual que en el proyecto original
EPOCAS_ETAPA_1 = 2            # solo la cabeza (base congelada)
EPOCAS_ETAPA_2 = 10           # ajuste fino de toda la red (con EarlyStopping)
UMBRAL = 0.5
N_EJEMPLOS = 6                # imágenes de validación que se exportan para la demo web

# Prueba rápida: pocas imágenes y una época, para comprobar que todo el flujo funciona.
PRUEBA_RAPIDA = os.getenv("PRUEBA_RAPIDA", "0") == "1"
if PRUEBA_RAPIDA:
    EPOCAS_ETAPA_1, EPOCAS_ETAPA_2 = 1, 1

DIR_SALIDA = os.getenv("DIR_SALIDA", "artefactos")
DIR_CACHE = os.getenv("DIR_CACHE", "/tmp/aptos_512")

In [ ]:
import glob
import json
import random
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers, ops
from PIL import Image, ImageOps
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

random.seed(SEMILLA)
np.random.seed(SEMILLA)
keras.utils.set_random_seed(SEMILLA)

os.makedirs(DIR_SALIDA, exist_ok=True)
print("TensorFlow", tf.__version__, "· Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "ninguna (irá MUY lento)")

## 1. Datos
APTOS 2019 etiqueta cada retinografía con un grado de 0 a 4. Como en el proyecto original, el problema se
plantea como **clasificación binaria**: `0` = sin retinopatía, `1` = cualquier grado de retinopatía.

In [ ]:
def localizar_dataset():
    """Busca el train.csv de APTOS (columnas id_code, diagnosis) y su carpeta de imágenes."""
    raices = [os.getenv("DATA_DIR"), "/kaggle/input", "data"]
    for raiz in filter(None, raices):
        for csv in glob.glob(os.path.join(raiz, "**", "train.csv"), recursive=True):
            if {"id_code", "diagnosis"} <= set(pd.read_csv(csv, nrows=1).columns):
                carpeta = os.path.join(os.path.dirname(csv), "train_images")
                if os.path.isdir(carpeta):
                    return csv, carpeta
    raise FileNotFoundError(
        "No se encontró APTOS 2019. En Kaggle: Add input → 'APTOS 2019 Blindness Detection'. "
        "En local: define DATA_DIR con la carpeta que contiene train.csv y train_images/."
    )


RUTA_CSV, CARPETA_IMAGENES = localizar_dataset()
df = pd.read_csv(RUTA_CSV)
df["ruta"] = df["id_code"].map(lambda i: os.path.join(CARPETA_IMAGENES, f"{i}.png"))
df["level"] = (df["diagnosis"] > 0).astype(int)

if PRUEBA_RAPIDA:
    df = df.groupby("level", group_keys=False).sample(n=24, random_state=SEMILLA).reset_index(drop=True)

print(f"{len(df)} imágenes · {df['level'].mean():.1%} con retinopatía")
df.head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
grados = ["0 · Sano", "1 · Leve", "2 · Moderada", "3 · Severa", "4 · Proliferativa"]
conteo = df["diagnosis"].value_counts().reindex(range(5), fill_value=0)
ax1.bar(grados, conteo, color=["#0f766e", "#f0a35e", "#e07a3f", "#d9542b", "#9b2c12"])
ax1.set_title("Grado original (APTOS)")
ax1.tick_params(axis="x", rotation=25)
binario = df["level"].value_counts().reindex([0, 1], fill_value=0)
ax2.bar(["0 · Sano", "1 · Retinopatía"], binario, color=["#0f766e", "#d9542b"])
ax2.set_title("Etiqueta binaria (la que aprende el modelo)")
plt.tight_layout()
plt.show()

## 2. Preprocesado
Las fotos originales son enormes (hasta 4 000 px) y traen un marco negro alrededor de la retina. Se recortan y
redimensionan **una sola vez** a 512 × 512 y se guardan en disco, para que cada época no tenga que decodificarlas de nuevo.

> ⚠️ Estas funciones son **idénticas** a las de `src/retinopatia/preprocess.py`, que es lo que usa la aplicación web.
> Si cambias una, cambia la otra: el modelo debe recibir en producción lo mismo que vio al entrenar.

In [ ]:
UMBRAL_FONDO = 10


def recortar_bordes(imagen, umbral=UMBRAL_FONDO):
    """Recorta el marco negro que rodea a la retina."""
    gris = np.asarray(imagen.convert("L"))
    mascara = gris > umbral
    if not mascara.any():
        return imagen
    filas = np.flatnonzero(mascara.any(axis=1))
    columnas = np.flatnonzero(mascara.any(axis=0))
    return imagen.crop((columnas[0], filas[0], columnas[-1] + 1, filas[-1] + 1))


def normalizar_imagen(imagen, tamano=TAMANO):
    """RGB, sin bordes negros y redimensionada a tamano x tamano."""
    imagen = ImageOps.exif_transpose(imagen).convert("RGB")
    imagen = recortar_bordes(imagen)
    return imagen.resize((tamano, tamano), Image.BILINEAR)


def cachear(ruta):
    destino = os.path.join(DIR_CACHE, Path(ruta).stem + ".png")
    if not os.path.exists(destino):
        with Image.open(ruta) as imagen:
            normalizar_imagen(imagen).save(destino, compress_level=1)
    return destino


os.makedirs(DIR_CACHE, exist_ok=True)
with ThreadPoolExecutor(max_workers=os.cpu_count()) as grupo:
    df["cache"] = list(grupo.map(cachear, df["ruta"]))
print("Imágenes preprocesadas en", DIR_CACHE)

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(12, 6))
for eje, (_, fila) in zip(ejes.T, df.sample(4, random_state=SEMILLA).iterrows()):
    eje[0].imshow(Image.open(fila["ruta"]))
    eje[0].set_title(f"Original · grado {fila['diagnosis']}")
    eje[1].imshow(Image.open(fila["cache"]))
    eje[1].set_title("Recortada 512 × 512")
    for e in eje:
        e.axis("off")
plt.tight_layout()
plt.show()

## 3. Partición y generadores
80 % entrenamiento / 20 % validación, **estratificada por grado** y con semilla fija. La aumentación (volteos) se
aplica **solo al entrenamiento**: en el original también afectaba a la validación.

*Limitación:* APTOS no publica identificadores de paciente, así que no se puede garantizar que los dos ojos de
una misma persona caigan en el mismo lado de la partición.

In [ ]:
df_train, df_val = train_test_split(
    df, test_size=0.2, stratify=df["level" if PRUEBA_RAPIDA else "diagnosis"], random_state=SEMILLA
)
print(f"Entrenamiento: {len(df_train)} · Validación: {len(df_val)}")


def leer(ruta, etiqueta):
    imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
    imagen = tf.ensure_shape(imagen, (TAMANO, TAMANO, 3))
    return tf.cast(imagen, tf.float32), etiqueta  # RGB 0-255: la normalización va dentro del modelo


def aumentar(imagen, etiqueta):
    imagen = tf.image.random_flip_left_right(imagen)
    imagen = tf.image.random_flip_up_down(imagen)
    return imagen, etiqueta


def crear_dataset(tabla, entrenamiento):
    ds = tf.data.Dataset.from_tensor_slices((tabla["cache"].values, tabla["level"].values.astype("float32")))
    if entrenamiento:
        ds = ds.shuffle(len(tabla), seed=SEMILLA)
    ds = ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE)
    if entrenamiento:
        ds = ds.map(aumentar, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(LOTE).prefetch(tf.data.AUTOTUNE)


ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val = crear_dataset(df_val, entrenamiento=False)

## 4. Modelo
La misma arquitectura del proyecto original: **ResNet50** (ImageNet) sin su clasificador, seguida de
promedio global → `Dropout(0.2)` → `Dense(256, relu)` → `Dense(1, sigmoid)`.

La única novedad es la primera capa: aplica la normalización con la que se entrenó ResNet50 (RGB → BGR y resta de
la media de ImageNet). Al ir *dentro* del modelo, es imposible que entrenamiento e inferencia se desincronicen.

In [ ]:
@keras.saving.register_keras_serializable(package="retinopatia")
class PreprocesadoResNet(layers.Layer):
    """Equivale a keras.applications.resnet.preprocess_input, pero viaja con el modelo."""

    MEDIA_BGR = (103.939, 116.779, 123.68)

    def call(self, imagenes):
        bgr = ops.flip(imagenes, axis=-1)
        return bgr - ops.convert_to_tensor(self.MEDIA_BGR, dtype=bgr.dtype)


base = keras.applications.ResNet50(weights="imagenet", include_top=False, input_shape=(TAMANO, TAMANO, 3))

entrada = keras.Input(shape=(TAMANO, TAMANO, 3), name="imagen")
mapas = base(PreprocesadoResNet(name="preprocesado")(entrada))   # (16, 16, 2048)
x = layers.GlobalAveragePooling2D(name="promedio")(mapas)
x = layers.Dropout(0.2)(x)
densa = layers.Dense(256, activation="relu", name="densa")
salida = layers.Dense(1, activation="sigmoid", name="probabilidad")
oculta = densa(x)
model = keras.Model(entrada, salida(oculta), name="retinopatia_resnet50")


def compilar(tasa):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=tasa),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )

## 5. Etapa 1 — entrenar solo la cabeza
Con ResNet50 congelada, las capas nuevas (con pesos aleatorios) aprenden sin estropear lo que la base ya sabe.

In [ ]:
base.trainable = False
compilar(1e-3)
model.summary(show_trainable=True, line_length=100, layer_range=["promedio", "probabilidad"])

history = model.fit(ds_train, epochs=EPOCAS_ETAPA_1, validation_data=ds_val, verbose=2).history

## 6. Etapa 2 — ajuste fino de toda la red
Se descongela todo y se baja la tasa de aprendizaje diez veces. `EarlyStopping` recupera los mejores pesos y
`ReduceLROnPlateau` reduce la tasa si la validación se estanca (los mismos *callbacks* del original).

In [ ]:
base.trainable = True
compilar(1e-4)

es = keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=4, restore_best_weights=True, verbose=1)
rlrop = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", patience=2, factor=0.5, min_lr=1e-6, verbose=1)

history_mejorado = model.fit(
    ds_train, epochs=EPOCAS_ETAPA_2, validation_data=ds_val, callbacks=[es, rlrop], verbose=2
).history

model.save_weights(os.path.join(DIR_SALIDA, "modelo.weights.h5"))  # copia de seguridad de los pesos

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for eje, clave, titulo in [(ax1, "loss", "Pérdida"), (ax2, "accuracy", "Exactitud")]:
    entrenamiento = history[clave] + history_mejorado[clave]
    validacion = history["val_" + clave] + history_mejorado["val_" + clave]
    epocas = range(1, len(entrenamiento) + 1)
    eje.plot(epocas, entrenamiento, label="Entrenamiento", color="#0e1b2a")
    eje.plot(epocas, validacion, label="Validación", color="#d9542b")
    eje.axvline(EPOCAS_ETAPA_1 + 0.5, ls="--", color="gray", lw=1)
    eje.set_title(titulo)
    eje.set_xlabel("Época")
    eje.legend()
plt.tight_layout()
plt.show()

## 7. Evaluación
En un problema de cribado médico la exactitud no basta: importa sobre todo la **sensibilidad** (no dejar pasar
ojos enfermos). Se reportan también especificidad, precisión, F1 y el AUC-ROC.

In [ ]:
y_real = df_val["level"].values
y_prob = model.predict(ds_val, verbose=0).ravel()
y_pred = (y_prob >= UMBRAL).astype(int)

vn, fp, fn, vp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
division = lambda a, b: float(a / b) if b else 0.0
metricas = {
    "dataset": "APTOS 2019",
    "fecha": date.today().isoformat(),
    "prueba_rapida": PRUEBA_RAPIDA,
    "n_train": int(len(df_train)),
    "n_val": int(len(df_val)),
    "umbral": UMBRAL,
    "auc": float(roc_auc_score(y_real, y_prob)),
    "accuracy": division(vp + vn, len(y_real)),
    "sensibilidad": division(vp, vp + fn),
    "especificidad": division(vn, vn + fp),
    "precision": division(vp, vp + fp),
    "matriz_confusion": {"vn": int(vn), "fp": int(fp), "fn": int(fn), "vp": int(vp)},
    "historial": {
        "etapa1": {k: [float(v) for v in serie] for k, serie in history.items() if "auc" not in k},
        "etapa2": {k: [float(v) for v in serie] for k, serie in history_mejorado.items() if k.split("_")[-1] in ("loss", "accuracy")},
    },
}
metricas["f1"] = division(2 * metricas["precision"] * metricas["sensibilidad"], metricas["precision"] + metricas["sensibilidad"])

for clave in ("auc", "accuracy", "sensibilidad", "especificidad", "precision", "f1"):
    print(f"{clave:>14}: {metricas[clave]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow([[vn, fp], [fn, vp]], cmap="Oranges")
for (i, j), valor in np.ndenumerate([[vn, fp], [fn, vp]]):
    ax1.text(j, i, valor, ha="center", va="center", fontsize=16)
ax1.set_xticks([0, 1], ["Predijo sano", "Predijo RD"])
ax1.set_yticks([0, 1], ["Sano", "Con RD"])
ax1.set_title("Matriz de confusión")
tfp, tvp, _ = roc_curve(y_real, y_prob)
ax2.plot(tfp, tvp, color="#d9542b", label=f"AUC = {metricas['auc']:.3f}")
ax2.plot([0, 1], [0, 1], ls="--", color="gray")
ax2.set_xlabel("1 − especificidad")
ax2.set_ylabel("Sensibilidad")
ax2.set_title("Curva ROC")
ax2.legend()
plt.tight_layout()
plt.show()

## 8. Grad-CAM dentro del modelo
Grad-CAM pondera cada mapa de características $A_k$ por el gradiente de la salida respecto a él. En esta
arquitectura ese gradiente tiene **forma cerrada**, porque entre los mapas y la salida solo hay un promedio y dos capas densas:

$$\alpha = W_1^\top \left(w_2 \odot \mathbb{1}[h > 0]\right), \qquad \text{CAM} = \mathrm{ReLU}\Big(\sum_k \alpha_k A_k\Big)$$

Así el mapa se calcula con operaciones normales, se exporta a ONNX junto con la probabilidad, y la aplicación web
no necesita TensorFlow para explicar sus predicciones.

In [ ]:
class GradCAM(layers.Layer):
    """Grad-CAM exacto (respecto al logit) para una cabeza promedio → densa ReLU → densa."""

    def __init__(self, densa, salida, **kwargs):
        super().__init__(**kwargs)
        self.densa, self.salida = densa, salida

    def call(self, entradas):
        mapas, oculta = entradas
        activas = ops.cast(oculta > 0, mapas.dtype)                                   # (N, 256)
        alfa = ops.matmul(activas * ops.transpose(self.salida.kernel), ops.transpose(self.densa.kernel))  # (N, 2048)
        return ops.relu(ops.sum(mapas * alfa[:, None, None, :], axis=-1))             # (N, 16, 16)


modelo_export = keras.Model(
    entrada,
    [model.output, GradCAM(densa, salida, name="gradcam")([mapas, oculta])],
    name="retinopatia_con_gradcam",
)

In [ ]:
def superponer(eje, ruta, mapa, titulo):
    imagen = Image.open(ruta)
    mapa = mapa / mapa.max() if mapa.max() > 0 else mapa
    calor = np.asarray(Image.fromarray(np.uint8(mapa * 255)).resize(imagen.size, Image.BICUBIC))
    eje.imshow(imagen)
    eje.imshow(calor, cmap="plasma", alpha=0.45)
    eje.set_title(titulo, fontsize=10)
    eje.axis("off")


muestra = df_val[df_val["level"] == 1].head(4)
lote = np.stack([np.asarray(Image.open(r), dtype="float32") for r in muestra["cache"]])
probs, cams = modelo_export.predict(lote, verbose=0)
fig, ejes = plt.subplots(1, len(muestra), figsize=(3.5 * len(muestra), 3.8))
for eje, ruta, grado, p, cam in zip(np.atleast_1d(ejes), muestra["cache"], muestra["diagnosis"], probs.ravel(), cams):
    superponer(eje, ruta, cam, f"grado real {grado} · p = {p:.2f}")
plt.tight_layout()
plt.show()

## 9. Exportar a ONNX
ONNX Runtime pesa ~15 MB frente a los ~600 MB de TensorFlow: es lo que hace viable un despliegue gratuito.
Tras exportar se comprueba que ONNX y Keras dan **el mismo resultado**.

In [ ]:
RUTA_ONNX = os.path.join(DIR_SALIDA, "modelo.onnx")

try:
    modelo_export.export(RUTA_ONNX, format="onnx")            # Keras ≥ 3.8
except Exception as error:                                     # noqa: BLE001
    print("Exportación nativa no disponible:", error, "\n→ usando SavedModel + tf2onnx")
    import subprocess
    import sys

    modelo_export.export("/tmp/modelo_tf")                     # SavedModel
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf2onnx"], check=True)
    subprocess.run(
        [sys.executable, "-m", "tf2onnx.convert", "--saved-model", "/tmp/modelo_tf", "--output", RUTA_ONNX, "--opset", "17"],
        check=True,
    )

try:
    import onnxruntime as ort
except ImportError:
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=True)
    import onnxruntime as ort

sesion = ort.InferenceSession(RUTA_ONNX, providers=["CPUExecutionProvider"])
nombres = [s.name for s in sorted(sesion.get_outputs(), key=lambda s: len(s.shape))]
probs_onnx, cams_onnx = sesion.run(nombres, {sesion.get_inputs()[0].name: lote})

print("Diferencia máxima en probabilidad:", float(np.abs(probs_onnx - probs).max()))
print("Diferencia máxima relativa en Grad-CAM:", float(np.abs(cams_onnx - cams).max() / (np.abs(cams).max() + 1e-9)))
assert np.allclose(probs_onnx, probs, atol=1e-3), "ONNX y Keras no coinciden"
print(f"✔ modelo.onnx verificado ({os.path.getsize(RUTA_ONNX) / 1e6:.0f} MB)")

## 10. Artefactos para la aplicación web
- `modelo.onnx` — el modelo (probabilidad + Grad-CAM).
- `metrics.json` — métricas y curvas que muestra la web.
- `ejemplos/` — unas pocas imágenes de validación para probar la demo con un clic.

In [ ]:
dir_ejemplos = os.path.join(DIR_SALIDA, "ejemplos")
os.makedirs(dir_ejemplos, exist_ok=True)

df_val = df_val.assign(prob=y_prob)
grados_deseados = [0, 0, 1, 2, 3, 4][:N_EJEMPLOS]
elegidos, usados = [], set()
for grado in grados_deseados:
    candidatos = df_val[(df_val["diagnosis"] == grado) & ~df_val["id_code"].isin(usados)]
    if candidatos.empty:
        continue
    fila = candidatos.sample(1, random_state=SEMILLA).iloc[0]   # al azar: incluye aciertos y posibles fallos
    usados.add(fila["id_code"])
    archivo = f"{fila['id_code']}.jpg"
    Image.open(fila["cache"]).save(os.path.join(dir_ejemplos, archivo), quality=90)
    elegidos.append({"archivo": archivo, "etiqueta": int(fila["level"]), "grado": int(fila["diagnosis"])})

with open(os.path.join(dir_ejemplos, "ejemplos.json"), "w", encoding="utf-8") as f:
    json.dump({"fuente": "APTOS 2019 (conjunto de validación)", "ejemplos": elegidos}, f, ensure_ascii=False, indent=2)
with open(os.path.join(DIR_SALIDA, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

# Todo lo que necesita la web, en un único archivo para descargar (sin la copia de los pesos de Keras).
with zipfile.ZipFile("artefactos.zip", "w") as z:
    for ruta in Path(DIR_SALIDA).rglob("*"):
        if ruta.is_file() and ruta.suffix != ".h5":
            z.write(ruta, ruta.relative_to(DIR_SALIDA))
print("Listo → artefactos.zip:", f"{os.path.getsize('artefactos.zip') / 1e6:.0f} MB")

## Siguiente paso
Descarga **`artefactos.zip`** (panel *Output* de Kaggle), descomprímelo dentro de `models/` en el repositorio y
sigue el README: `models/modelo.onnx`, `models/metrics.json` y `models/ejemplos/`.